In [1]:
import os
import sys
import pandas as pd
import re
from collections import Counter

# 환경 설정
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
sys.path.append(project_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 데이터 로드
train_df = pd.read_csv(os.path.join(project_dir, 'data', 'train.csv'))
val_df = pd.read_csv(os.path.join(project_dir, 'data', 'dev.csv'))
test_df = pd.read_csv(os.path.join(project_dir, 'data', 'test.csv'))

# 전체를 하나의 dict로 관리
datasets = {'train': train_df, 'dev': val_df, 'test': test_df}

# 지시어+명사 추출

In [6]:
from collections import Counter
from konlpy.tag import Okt

# 지시어/지칭어 사전 정의
DEICTIC_SET = set([
    # 관형사/지시어
    '이', '그', '저', '여기', '거기', '저기', '이쪽', '그쪽', '저쪽',
    '이런', '그런', '저런', '이곳', '그곳', '저곳',
    '이분', '그분', '저분', '이것', '그것', '저것',
    '이사람', '그사람', '저사람', '이놈', '그놈', '저놈',
    '이년', '그년', '저년', '이자', '그자', '저자', '이이', '그이', '저이',
    # 대명사/지칭어
    '내', '나', '너', '우리', '너희', '당신', '자기', '저희',
    '걔', '쟤', '그녀', '이분들', '그분들', '저분들', '이놈들', '그놈들', '저놈들', '얘', '얘네', '얘들아',
    '그대', '임', '자네',
    # 지시부사/의문사
    '이렇게', '그렇게', '저렇게', '이만큼', '그만큼', '저만큼',
    '이따', '무엇', '누구', '언제', '어디', '어떻게', '얼마', '뭐', '누가', '어느', '어떤', '왜',
    # 복합형은 붙여쓰기/띄어쓰기 모두 탐지
    '그 사람', '이 사람', '저 사람', '그분', '이분', '저분', '그거', '이거', '저거', '그녀', '그놈', '그년',
    '저 친구', '그 친구', '이 친구', '그 아이', '이 아이', '저 아이', '이것들', '그것들', '저것들',
    '그때', '이때', '저때', '그날', '이날', '저날', '그쪽', '이쪽', '저쪽', '이곳', '그곳', '저곳',
    '이런 것', '그런 것', '저런 것'
])

# n-gram 추출 함수
def extract_ngrams(sentence, n=2):
    tokens = sentence.split()
    grams = [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    # 붙여쓰기 케이스도 추가
    for i in range(len(tokens)-n+1):
        grams.append(''.join(tokens[i:i+n]))
    return grams

# 전체 n-gram 추출
def get_all_ngrams(df, n):
    ngrams = []
    for sent in df['dialogue']:
        ngrams += extract_ngrams(sent, n)
    return ngrams

# 품사 태깅(명사 판단)
def is_noun(word, okt):
    try:
        return any([tag in ['Noun', 'ProperNoun'] for _, tag in okt.pos(word)])
    except:
        return False

# 지시어+명사 패턴만 필터링
def filter_deictic_noun_patterns(ngrams, okt, n=2):
    patterns = []
    for ngram in ngrams:
        words = ngram.replace('"', '').replace(',', '').strip().split()
        # 2-gram
        if n == 2:
            if len(words) == 2:
                # Case 1: 띄어쓰기(그 사람)
                if words[0] in DEICTIC_SET and is_noun(words[1], okt):
                    patterns.append(ngram)
            # Case 2: 붙여쓰기(그사람)
            elif len(words) == 1 and len(words[0]) >= 3:
                for deictic in DEICTIC_SET:
                    if words[0].startswith(deictic):
                        noun_candidate = words[0][len(deictic):]
                        if noun_candidate and is_noun(noun_candidate, okt):
                            patterns.append(ngram)
        # 3-gram: 지시어/관형사/부사 + 명사 (이런 좋은 사람 등)
        elif n == 3:
            if len(words) == 3:
                if words[0] in DEICTIC_SET and is_noun(words[2], okt):
                    patterns.append(ngram)
    return patterns

# 실행
okt = Okt()
results = []

for n in [2, 3]:
    ngrams = get_all_ngrams(train_df, n)
    filtered = filter_deictic_noun_patterns(ngrams, okt, n=n)
    counter = Counter(filtered)
    for k, v in counter.most_common(500):  # 상위 500개까지
        results.append({'pattern': k, 'count': v, 'n': n})

# 결과 저장(전체)
result_df = pd.DataFrame(results)
result_df.to_csv('deictic_ngram_patterns.csv', index=False)

# 지시+명사 패턴만 따로 저장
result_df[result_df['count'] >= 1][['pattern', 'count', 'n']].to_csv('filtered_deictic_patterns.csv', index=False)

In [8]:
from collections import Counter
from konlpy.tag import Okt

# 지시/대명/지칭어 리스트 정의 (위에서 확장한 DEICTIC_SET 활용)
DEICTIC_SET = [
    # 관형사/지시어
    '이', '그', '저', '여기', '거기', '저기', '이쪽', '그쪽', '저쪽',
    '이런', '그런', '저런', '이곳', '그곳', '저곳',
    '이분', '그분', '저분', '이것', '그것', '저것',
    '이사람', '그사람', '저사람', '이놈', '그놈', '저놈',
    '이년', '그년', '저년', '이자', '그자', '저자', '이이', '그이', '저이',
    # 대명사/지칭어
    '내', '나', '너', '우리', '너희', '당신', '자기', '저희',
    '걔', '쟤', '그녀', '이분들', '그분들', '저분들', '이놈들', '그놈들', '저놈들', '얘', '얘네', '얘들아',
    '그대', '임', '자네',
    # 지시부사/의문사
    '이렇게', '그렇게', '저렇게', '이만큼', '그만큼', '저만큼',
    '이따', '무엇', '누구', '언제', '어디', '어떻게', '얼마', '뭐', '누가', '어느', '어떤', '왜',
    # 복합형
    '그 사람', '이 사람', '저 사람', '그분', '이분', '저분', '그거', '이거', '저거', '그녀', '그놈', '그년',
    '저 친구', '그 친구', '이 친구', '그 아이', '이 아이', '저 아이', '이것들', '그것들', '저것들',
    '그때', '이때', '저때', '그날', '이날', '저날', '그쪽', '이쪽', '저쪽', '이곳', '그곳', '저곳',
    '이런 것', '그런 것', '저런 것'
]
okt = Okt()

# n-gram + 예시 추출 (2-gram/3-gram)
def extract_ngrams(sentence, n=2):
    tokens = sentence.split()
    grams = [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    for i in range(len(tokens)-n+1):
        grams.append(''.join(tokens[i:i+n]))
    return grams

def get_all_ngrams(df, n):
    ngrams = []
    idx = []
    for idx_, sent in zip(df['fname'], df['dialogue']):
        grams = extract_ngrams(sent, n)
        ngrams.extend(grams)
        idx.extend([idx_]*len(grams))
    return ngrams, idx

# 지시/대명/지칭어 포함 패턴/예시 추출 및 저장
def collect_deictic_patterns(df, n, set_deictic):
    ngrams, idxs = get_all_ngrams(df, n)
    patterns = []
    for ngram, idx in zip(ngrams, idxs):
        for d in set_deictic:
            if d in ngram:
                patterns.append((ngram, idx))
    return patterns

results = []
for n, name, df in zip([2, 3], ["train", "dev", "test"], [train_df, val_df, test_df]):
    patterns = collect_deictic_patterns(df, n, DEICTIC_SET)
    c = Counter([x[0] for x in patterns])
    # 패턴별 예시 2개 저장
    example_dict = {}
    for p, idx in patterns:
        if p not in example_dict:
            example_dict[p] = []
        if len(example_dict[p]) < 2:
            row = df[df['fname']==idx].iloc[0]
            example_dict[p].append(row['dialogue'][:120])
    # csv로
    rows = []
    for pat, cnt in c.most_common(300):
        examples = example_dict.get(pat, [])
        rows.append({'ngram': pat, 'count': cnt, 'n': n, 'example1': examples[0] if examples else '', 'example2': examples[1] if len(examples)>1 else ''})
    pd.DataFrame(rows).to_csv(f'deictic_patterns_{name}.csv', index=False)

print('완료: 각 split별 지시/대명/지칭어 n-gram/예시 csv 저장됨')

완료: 각 split별 지시/대명/지칭어 n-gram/예시 csv 저장됨


### "그 사람", "이 사람", "저 사람" 등 이 들어간 모든 대화 라인 예시 추출해서 확인

In [ ]:
import re

deictic_keys = [
    # 관형사/지시어
    '이', '그', '저', '여기', '거기', '저기', '이쪽', '그쪽', '저쪽',
    '이런', '그런', '저런', '이곳', '그곳', '저곳',
    '이분', '그분', '저분', '이것', '그것', '저것',
    '이사람', '그사람', '저사람', '이놈', '그놈', '저놈',
    '이년', '그년', '저년', '이자', '그자', '저자', '이이', '그이', '저이',
    # 대명사/지칭어
    '내', '나', '너', '우리', '너희', '당신', '자기', '저희',
    '걔', '쟤', '그녀', '이분들', '그분들', '저분들', '이놈들', '그놈들', '저놈들', '얘', '얘네', '얘들아',
    '그대', '임', '자네',
    # 지시부사/의문사
    '이렇게', '그렇게', '저렇게', '이만큼', '그만큼', '저만큼',
    '이따', '무엇', '누구', '언제', '어디', '어떻게', '얼마', '뭐', '누가', '어느', '어떤', '왜',
    # 복합형
    '그 사람', '이 사람', '저 사람', '그분', '이분', '저분', '그거', '이거', '저거', '그녀', '그놈', '그년',
    '저 친구', '그 친구', '이 친구', '그 아이', '이 아이', '저 아이', '이것들', '그것들', '저것들',
    '그때', '이때', '저때', '그날', '이날', '저날', '그쪽', '이쪽', '저쪽', '이곳', '그곳', '저곳',
    '이런 것', '그런 것', '저런 것'
]

for name, df in zip(['train', 'dev', 'test'], [train_df, val_df, test_df]):
    examples = []
    for i, row in df.iterrows():
        lines = str(row['dialogue']).split('\n')
        for l in lines:
            if any(key in l for key in deictic_keys):
                examples.append({'fname': row['fname'], 'line': l})
    pd.DataFrame(examples).to_csv(f'deictic_lines_{name}.csv', index=False)
    
    print(f'{name}: {len(examples)}개 추출됨')

train: 44746개 추출됨
dev: 1742개 추출됨
test: 1823개 추출됨


---

### 1. 전제 및 가정
- 데이터 구조:
fname, dialogue (1개 대화=여러줄, 각 줄은 #PersonN#: 형태)

- 지시어 패턴:
"그 사람", "이 사람", "저 사람" 등
<br>기타 복수/불특정/직책+이름형은 추가 규칙 후처리 (EDA 기반)

- PersonN 규칙:
등장 순서대로 인원 파악 (#Person1#, #Person2#, …)

In [5]:
import re
import pandas as pd
from konlpy.tag import Okt
from collections import Counter

okt = Okt()

def extract_deictic_noun_patterns(text):
    patterns = []
    for match in re.finditer(r'(이|그|저)\s?([가-힣]{1,6})', text):
        word = match.group(2)
        pos_list = okt.pos(word)
        if pos_list and pos_list[0][1] in ['Noun', 'ProperNoun']:
            patterns.append(match.group(0))
    return patterns

def get_num_speakers(dialogue):
    matches = re.findall(r'#Person(\d+)#', dialogue)
    return max(map(int, matches)) if matches else 0

def get_context(lines, idx, window=2):
    start = max(0, idx-window)
    end = min(len(lines), idx+window+1)
    return '\n'.join(lines[start:end])

def collect_deictic_patterns_with_context(df, split_name):
    pattern_counter = Counter()
    pattern_examples = {}
    pattern_num_speakers = {}

    for _, row in df.iterrows():
        dialogue = str(row['dialogue'])
        lines = dialogue.split('\n')
        num_speakers = get_num_speakers(dialogue)

        for i, line in enumerate(lines):
            for pat in extract_deictic_noun_patterns(line):
                pattern_counter[pat] += 1
                if pat not in pattern_examples:
                    pattern_examples[pat] = []
                    pattern_num_speakers[pat] = []
                # 앞뒤 2줄 context 추출 (중복 예시 방지)
                context = get_context(lines, i, window=2)
                if context not in pattern_examples[pat] and len(pattern_examples[pat]) < 2:
                    pattern_examples[pat].append(context)
                    pattern_num_speakers[pat].append(num_speakers)

    rows = []
    for pat, cnt in pattern_counter.most_common():
        examples = pattern_examples.get(pat, [])
        nums = pattern_num_speakers.get(pat, [])
        rows.append({
            'pattern': pat,
            'count': cnt,
            'context1': examples[0] if len(examples) > 0 else '',
            'context2': examples[1] if len(examples) > 1 else '',
            'num_speakers1': nums[0] if len(nums) > 0 else '',
            'num_speakers2': nums[1] if len(nums) > 1 else '',
        })
    result_df = pd.DataFrame(rows)
    result_df.to_csv(f'deictic_noun_patterns_{split_name}.csv', index=False)
    print(f"[{split_name}] {len(result_df)}개 패턴 저장 완료.")
    return result_df

# --- split별 실행 ---
all_pattern_dfs = {}
for split_name, df in zip(['train', 'dev', 'test'], [train_df, val_df, test_df]):
    all_pattern_dfs[split_name] = collect_deictic_patterns_with_context(df, split_name)

# --- 통합(전체) csv 저장 ---
all_patterns_df = pd.concat(all_pattern_dfs.values(), ignore_index=True)
all_patterns_agg = all_patterns_df.groupby('pattern').agg({
    'count': 'sum',
    'context1': 'first',
    'context2': 'first',
    'num_speakers1': 'first',
    'num_speakers2': 'first'
}).sort_values('count', ascending=False).reset_index()
all_patterns_agg.to_csv('deictic_noun_patterns_all.csv', index=False)
print(f"[ALL] 통합 후보 {len(all_patterns_agg)}개 저장 완료.")


[train] 10458개 패턴 저장 완료.
[dev] 1047개 패턴 저장 완료.
[test] 1099개 패턴 저장 완료.
[ALL] 통합 후보 11085개 저장 완료.
